# EDA Governance (INBOX → STG → DW control) — Evidencia Capítulo 6 (UCM)

**Archivo:** `EDA_GOVERNANCE_INBOX_STG_v2.ipynb`

**Alineación documental (fuente oficial):**
- Entrada INBOX: `/data_lake/02_INBOX/{LIC|OC}/{YYYY-MM}/` fileciteturn2file5L33-L36
- STG dinámico: `stg_lic_YYYY_MM`, `stg_oc_YYYY_MM` fileciteturn2file5L39-L44
- Control/decisiones: `dw.etl_control_cargas`, `dw.etl_decisiones_periodo` fileciteturn2file5L53-L58
- Decisiones RUN / NO-OP / EXCEPCION_FUENTE (gobernanza, no “fallo”) fileciteturn2file5L20-L25

**Propósito (Cap. 6 — Gobierno de datos):** evidencia reproducible de:
1) continuidad y volumetría de recepción (INBOX)  
2) trazabilidad de staging mensual (STG)  
3) estado/decisiones oficiales del pipeline (DW tablas de control)

**Reglas:** diagnóstico (no modifica datos). Exporta CSV/PNG a `outputs/EDA_GOV_INBOX_STG/`.


In [17]:
# =========================================
# 0 - Setup (paths, conexión, helpers)
# =========================================
import os, re
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

OUTPUT_DIR = Path("outputs/EDA_GOV_INBOX_STG")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def safe_to_csv(df: pd.DataFrame, filename: str):
    path = OUTPUT_DIR / filename
    df.to_csv(path, index=False, encoding="utf-8")
    print("OK ->", path.as_posix(), f"({len(df):,} filas)")

def safe_savefig(filename: str):
    path = OUTPUT_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    plt.close()
    print("OK ->", path.as_posix())

def human_mb(x_bytes: float) -> float:
    return x_bytes / (1024 * 1024)

def parse_period(s: str):
    m = re.search(r"(20\d{2})[-_](0[1-9]|1[0-2])", s)
    return f"{m.group(1)}-{m.group(2)}" if m else None

# Data Lake / INBOX
DL_ROOT = Path(os.getenv("DATA_LAKE_ROOT", "/media/engineer/WD450/ChileCompraDL"))
INBOX_DIR = Path(os.getenv("INBOX_DIR", str(DL_ROOT / "02_INBOX")))

# DB (Postgres)
def load_config():
    """Carga configuración de Postgres desde entorno y .env/.env.local si existe.

    IMPORTANTE (entorno local vs docker):
    - Si ejecutas el notebook en tu máquina (fuera de Docker), normalmente debes conectar a:
        host=localhost, port=5433 (según tu compose)
      usando variables NOTEBOOK_PG_HOST / NOTEBOOK_PG_PORT.
    - Si ejecutas dentro de un contenedor en la misma red docker, puedes usar host=chile-pg, port=5432.

    Prioridad de variables (de mayor a menor):
    1) NOTEBOOK_PG_HOST / NOTEBOOK_PG_PORT  (para notebooks locales)
    2) CHILE_PG_HOST / CHILE_PG_PORT
    3) POSTGRES_HOST / POSTGRES_PORT
    4) defaults: localhost / 5433
    """
    for env_file in [".env.local", ".env", ".env.example"]:
        p = Path(env_file)
        if p.exists():
            for line in p.read_text(encoding="utf-8", errors="ignore").splitlines():
                line = line.strip()
                if (not line) or line.startswith("#") or ("=" not in line):
                    continue
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

    host = os.getenv("NOTEBOOK_PG_HOST") or os.getenv("CHILE_PG_HOST") or os.getenv("POSTGRES_HOST") or "localhost"
    port = os.getenv("NOTEBOOK_PG_PORT") or os.getenv("CHILE_PG_PORT") or os.getenv("POSTGRES_PORT") or "5433"

    return {
        "PG_HOST": host,
        "PG_PORT": int(port),
        "PG_DB":   os.getenv("CHILE_PG_DB", os.getenv("POSTGRES_DB", "chilecompra")),
        "PG_USER": os.getenv("CHILE_PG_USER", os.getenv("POSTGRES_USER", "chile_user")),
        "PG_PASS": os.getenv("CHILE_PG_PASSWORD", os.getenv("POSTGRES_PASSWORD", "ChilePg_2024!")),
    }

def make_engine(cfg: dict):
    uri = f"postgresql+psycopg2://{cfg['PG_USER']}:{cfg['PG_PASS']}@{cfg['PG_HOST']}:{cfg['PG_PORT']}/{cfg['PG_DB']}"
    return create_engine(uri, pool_pre_ping=True)

def read_sql_df(q: str, params=None) -> pd.DataFrame:
    # SQLAlchemy 2.x: usar conexión explícita (evita OptionEngine.execute)
    with engine.connect() as conn:
        return pd.read_sql(text(q), conn, params=params or {})
    with engine.connect() as conn:
        df = pd.read_sql(text(q), conn, params=params)
    return df

cfg = load_config()
engine = make_engine(cfg)
print("Conexión (sin secretos):", {"host": cfg["PG_HOST"], "port": cfg["PG_PORT"], "db": cfg["PG_DB"], "user": cfg["PG_USER"]})
print("DL_ROOT:", DL_ROOT)
print("INBOX_DIR:", INBOX_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


# -------------------------
# Connectivity test (rápido)
# -------------------------
try:
    with engine.connect() as conn:
        v = conn.execute(text("SELECT 1")).scalar()
    print("DB connectivity: OK (SELECT 1 ->", v, ")")
except Exception as e:
    print("DB connectivity: ERROR ->", repr(e))
    print("Sugerencia (local notebook): export NOTEBOOK_PG_HOST=localhost y NOTEBOOK_PG_PORT=5433 (o el puerto host real).")


Conexión (sin secretos): {'host': 'localhost', 'port': 5433, 'db': 'chilecompra', 'user': 'chile_user'}
DL_ROOT: /media/engineer/WD450/ChileCompraDL
INBOX_DIR: /media/engineer/WD450/ChileCompraDL/02_INBOX
OUTPUT_DIR: /home/engineer/Documents/Proyectos/TFM/docs/evidence/Fase6_EDA/outputs/EDA_GOV_INBOX_STG
DB connectivity: OK (SELECT 1 -> 1 )


# 1 — Evidencia INBOX (continuidad y volumetría)

**Oficial:** `/data_lake/00_INBOX/{LIC|OC}/{YYYY-MM}/` fileciteturn2file5L33-L36

Figuras:
- **Figura 6.1** Archivos recibidos por mes (LIC vs OC) — barras agrupadas  
- **Figura 6.2** Tamaño total por mes (LIC + OC) — barras apiladas


In [18]:
# =========================================
# 1 - INBOX: lectura por estructura oficial
# =========================================
def list_inbox_files_struct(inbox_dir: Path):
    if not inbox_dir.exists():
        return pd.DataFrame([{"status":"NO_CONSTA_INBOX_DIR", "inbox_dir": str(inbox_dir)}])

    rows = []
    for dominio_dir in ["LIC", "OC"]:
        base = inbox_dir / dominio_dir
        if not base.exists():
            rows.append({"status":"NO_CONSTA_DOMINIO_DIR", "dominio": dominio_dir, "path": str(base)})
            continue
        for period_dir in base.iterdir():
            if not period_dir.is_dir():
                continue
            periodo = period_dir.name if re.fullmatch(r"20\d{2}-\d{2}", period_dir.name) else (parse_period(period_dir.name) or "NO_CONSTA")
            for p in period_dir.rglob("*"):
                if p.is_file():
                    rows.append({
                        "path": str(p),
                        "filename": p.name,
                        "dominio": dominio_dir,
                        "periodo": periodo,
                        "size_bytes": p.stat().st_size,
                        "size_mb": human_mb(p.stat().st_size),
                        "mtime": datetime.fromtimestamp(p.stat().st_mtime).isoformat(timespec="seconds"),
                        "status": "OK"
                    })
    return pd.DataFrame(rows)

df_inbox = list_inbox_files_struct(INBOX_DIR)
safe_to_csv(df_inbox, "tabla_inbox_archivos_detalle.csv")
display(df_inbox.head(20))

df_mes = (df_inbox[df_inbox.get("status","OK")=="OK"]
          .groupby(["periodo","dominio"], as_index=False)
          .agg(n_archivos=("filename","count"), size_mb=("size_mb","sum"))
          .sort_values(["periodo","dominio"]))
safe_to_csv(df_mes, "tabla_inbox_volumetria_mes.csv")
display(df_mes.head(20))

pivot_n = df_mes.pivot_table(index="periodo", columns="dominio", values="n_archivos", aggfunc="sum", fill_value=0).sort_index()
for c in ["LIC","OC"]:
    if c not in pivot_n.columns: pivot_n[c]=0
pivot_n = pivot_n[["LIC","OC"]]
x = np.arange(len(pivot_n.index)); w=0.42
plt.figure(figsize=(12,5))
plt.bar(x-w/2, pivot_n["LIC"].values, width=w, label="LIC")
plt.bar(x+w/2, pivot_n["OC"].values,  width=w, label="OC")
plt.xticks(x, pivot_n.index.tolist(), rotation=60, ha="right")
plt.ylabel("Nº archivos")
plt.title("Figura 6.1 — INBOX: archivos recibidos por mes (LIC vs OC)")
plt.legend()
safe_savefig("fig_6_1_inbox_archivos_por_mes.png")

pivot_s = df_mes.pivot_table(index="periodo", columns="dominio", values="size_mb", aggfunc="sum", fill_value=0).sort_index()
for c in ["LIC","OC"]:
    if c not in pivot_s.columns: pivot_s[c]=0.0
pivot_s = pivot_s[["LIC","OC"]]
lic = pivot_s["LIC"].values
oc  = pivot_s["OC"].values
x = np.arange(len(pivot_s.index))
plt.figure(figsize=(12,5))
plt.bar(x, lic, label="LIC (MB)")
plt.bar(x, oc, bottom=lic, label="OC (MB)")
plt.xticks(x, pivot_s.index.tolist(), rotation=60, ha="right")
plt.ylabel("Tamaño total (MB)")
plt.title("Figura 6.2 — INBOX: tamaño total por mes (apilado LIC + OC)")
plt.legend()
safe_savefig("fig_6_2_inbox_tamano_por_mes.png")


OK -> outputs/EDA_GOV_INBOX_STG/tabla_inbox_archivos_detalle.csv (30 filas)


,path,filename,dominio,periodo,size_bytes,size_mb,mtime,status
0,/media/engineer/WD450/ChileCompraDL/02_INBOX/L...,LIC_2024-09.csv,LIC,2024-09,306786738,292.574633,2025-03-28T09:55:04,OK
1,/media/engineer/WD450/ChileCompraDL/02_INBOX/L...,LIC_2024-10.csv,LIC,2024-10,379446203,361.868098,2026-01-16T11:49:25,OK
2,/media/engineer/WD450/ChileCompraDL/02_INBOX/L...,LIC_2024-11.csv,LIC,2024-11,863083512,823.100578,2025-07-25T13:39:20,OK
3,/media/engineer/WD450/ChileCompraDL/02_INBOX/L...,LIC_2024-12.csv,LIC,2024-12,573143214,546.591963,2025-06-24T09:53:58,OK
4,/media/engineer/WD450/ChileCompraDL/02_INBOX/L...,LIC_2025-01.csv,LIC,2025-01,155,0.000148,2025-11-13T14:42:32,OK
5,/media/engineer/WD450/ChileCompraDL/02_INBOX/L...,LIC_2025-02.csv,LIC,2025-02,1045864755,997.414355,2025-11-13T14:59:53,OK
6,/media/engineer/WD450/ChileCompraDL/02_INBOX/L...,LIC_2025-03.csv,LIC,2025-03,919858202,877.245142,2025-11-13T14:59:54,OK
7,/media/engineer/WD450/ChileCompraDL/02_INBOX/L...,LIC_2025-04.csv,LIC,2025-04,945603881,901.798135,2025-11-13T14:59:56,OK
8,/media/engineer/WD450/ChileCompraDL/02_INBOX/L...,LIC_2025-05.csv,LIC,2025-05,917834148,875.314854,2025-11-13T14:59:57,OK
9,/media/engineer/WD450/ChileCompraDL/02_INBOX/L...,LIC_2025-06.csv,LIC,2025-06,797273409,760.339173,2025-11-13T15:00:02,OK


OK -> outputs/EDA_GOV_INBOX_STG/tabla_inbox_volumetria_mes.csv (30 filas)


,periodo,dominio,n_archivos,size_mb
0,2024-09,LIC,1,292.574633
1,2024-09,OC,1,633.025342
2,2024-10,LIC,1,361.868098
3,2024-10,OC,1,812.679361
4,2024-11,LIC,1,823.100578
5,2024-11,OC,1,757.551730
6,2024-12,LIC,1,546.591963
7,2024-12,OC,1,666.363208
8,2025-01,LIC,1,0.000148
9,2025-01,OC,1,585.073648


OK -> outputs/EDA_GOV_INBOX_STG/fig_6_1_inbox_archivos_por_mes.png
OK -> outputs/EDA_GOV_INBOX_STG/fig_6_2_inbox_tamano_por_mes.png


# 2 — Evidencia STG mensual (existencia + volumen)

STG dinámico: `stg_lic_YYYY_MM`, `stg_oc_YYYY_MM` fileciteturn2file5L39-L44  
El flujo documental también referencia `public.stg_*` fileciteturn2file6L74-L80.  
Este notebook detecta **ambas variantes**.


In [19]:
# =========================================
# 2 - STG: detectar tablas en schema stg y/o public
# =========================================
def list_stg_tables_anyschema():
    q = '''
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_type='BASE TABLE'
      AND (
        (table_schema='stg' AND (table_name LIKE 'stg_lic_%' OR table_name LIKE 'stg_oc_%'))
        OR
        (table_schema='public' AND (table_name LIKE 'stg_lic_%' OR table_name LIKE 'stg_oc_%'))
      )
    ORDER BY table_schema, table_name
    '''
    return read_sql_df(q)

def infer_period_from_stg_table(table_name: str):
    m = re.search(r"(20\d{2})_(0[1-9]|1[0-2])$", table_name)
    return f"{m.group(1)}-{m.group(2)}" if m else None

def infer_domain_from_stg_table(table_name: str):
    n = table_name.lower()
    if n.startswith("stg_lic_"): return "LIC"
    if n.startswith("stg_oc_"):  return "OC"
    return "NO_CONSTA"

df_tabs = list_stg_tables_anyschema()
safe_to_csv(df_tabs, "tabla_stg_tablas_detectadas.csv")
display(df_tabs.head(20))

rows=[]
for _, r in df_tabs.iterrows():
    schema, t = r["table_schema"], r["table_name"]
    periodo = infer_period_from_stg_table(t) or "NO_CONSTA"
    dom = infer_domain_from_stg_table(t)
    n = read_sql_df(f"SELECT COUNT(*)::bigint AS n FROM {schema}.{t}").loc[0,"n"]
    rows.append({"periodo": periodo, "dominio": dom, "stg_table": f"{schema}.{t}", "n_registros": int(n), "status":"OK"})
df_stg = pd.DataFrame(rows).sort_values(["periodo","dominio"])
safe_to_csv(df_stg, "tabla_stg_registros_por_mes.csv")
display(df_stg.head(20))

df_plot = df_stg[df_stg["periodo"]!="NO_CONSTA"].groupby(["periodo","dominio"], as_index=False)["n_registros"].sum()
pivot = df_plot.pivot_table(index="periodo", columns="dominio", values="n_registros", aggfunc="sum", fill_value=0).sort_index()
for c in ["LIC","OC"]:
    if c not in pivot.columns: pivot[c]=0
pivot = pivot[["LIC","OC"]]
x = np.arange(len(pivot.index)); w=0.42
plt.figure(figsize=(12,5))
plt.bar(x-w/2, pivot["LIC"].values, width=w, label="LIC")
plt.bar(x+w/2, pivot["OC"].values,  width=w, label="OC")
plt.xticks(x, pivot.index.tolist(), rotation=60, ha="right")
plt.ylabel("Nº registros STG")
plt.title("Figura 6.3 — STG: registros cargados por mes (LIC vs OC)")
plt.legend()
safe_savefig("fig_6_3_stg_registros_por_mes.png")


OK -> outputs/EDA_GOV_INBOX_STG/tabla_stg_tablas_detectadas.csv (28 filas)


,table_schema,table_name
0,public,stg_lic_2024_09
1,public,stg_lic_2024_10
2,public,stg_lic_2024_11
3,public,stg_lic_2024_12
4,public,stg_lic_2025_01
5,public,stg_lic_2025_02
6,public,stg_lic_2025_03
7,public,stg_lic_2025_04
8,public,stg_lic_2025_05
9,public,stg_lic_2025_06


OK -> outputs/EDA_GOV_INBOX_STG/tabla_stg_registros_por_mes.csv (28 filas)


,periodo,dominio,stg_table,n_registros,status
0,2024-09,LIC,public.stg_lic_2024_09,178091,OK
14,2024-09,OC,public.stg_oc_2024_09,396516,OK
1,2024-10,LIC,public.stg_lic_2024_10,219827,OK
15,2024-10,OC,public.stg_oc_2024_10,508095,OK
2,2024-11,LIC,public.stg_lic_2024_11,530356,OK
16,2024-11,OC,public.stg_oc_2024_11,470710,OK
3,2024-12,LIC,public.stg_lic_2024_12,359662,OK
17,2024-12,OC,public.stg_oc_2024_12,411596,OK
4,2025-01,LIC,public.stg_lic_2025_01,2,OK
18,2025-01,OC,public.stg_oc_2025_01,360815,OK


OK -> outputs/EDA_GOV_INBOX_STG/fig_6_3_stg_registros_por_mes.png


# 3 — Evidencia oficial del estado del pipeline (DW control + decisiones)

Tablas oficiales:
- `dw.etl_control_cargas`, `dw.etl_decisiones_periodo` fileciteturn2file5L53-L58  
- Decisiones (RUN/NO_OP/EXCEPCION_FUENTE) fileciteturn2file5L20-L25


In [20]:
# =========================================
# 3 - DW control: etl_control_cargas + etl_decisiones_periodo
# =========================================
def table_exists(schema: str, table: str) -> bool:
    q = '''SELECT EXISTS(
            SELECT 1 FROM information_schema.tables 
            WHERE table_schema=:s AND table_name=:t
        )'''
    return bool(read_sql_df(q, {"s": schema, "t": table}).iloc[0,0])

has_ctrl = table_exists("dw","etl_control_cargas")
has_dec  = table_exists("dw","etl_decisiones_periodo")
print({"dw.etl_control_cargas": has_ctrl, "dw.etl_decisiones_periodo": has_dec})

if has_ctrl:
    df_ctrl = read_sql_df("SELECT * FROM dw.etl_control_cargas ORDER BY periodo")
    safe_to_csv(df_ctrl, "tabla_dw_control_cargas.csv")
    display(df_ctrl.head(20))
else:
    df_ctrl = pd.DataFrame([{"status":"NO_CONSTA_dw.etl_control_cargas"}])
    safe_to_csv(df_ctrl, "tabla_dw_control_cargas_status.csv")
    display(df_ctrl)

if has_dec:
    df_dec = read_sql_df("SELECT * FROM dw.etl_decisiones_periodo ORDER BY periodo")
    safe_to_csv(df_dec, "tabla_dw_decisiones_periodo.csv")
    display(df_dec.head(20))
else:
    df_dec = pd.DataFrame([{"status":"NO_CONSTA_dw.etl_decisiones_periodo"}])
    safe_to_csv(df_dec, "tabla_dw_decisiones_periodo_status.csv")
    display(df_dec)

# Figura 6.5 — decisiones por periodo (apilado)
if has_dec:
    cols = [c.lower() for c in df_dec.columns]
    if "periodo" in cols and "decision" in cols:
        periodo_col = df_dec.columns[cols.index("periodo")]
        decision_col = df_dec.columns[cols.index("decision")]
        d = df_dec.copy()
        d["periodo"] = d[periodo_col].astype(str)
        d["decision"] = d[decision_col].astype(str).str.upper()
        ts = d.groupby(["periodo","decision"], as_index=False).size().rename(columns={"size":"n"})
        pivot = ts.pivot_table(index="periodo", columns="decision", values="n", aggfunc="sum", fill_value=0).sort_index()
        x=np.arange(len(pivot.index))
        bottom = np.zeros(len(x))
        plt.figure(figsize=(12,5))
        for dec_name in pivot.columns.tolist():
            vals = pivot[dec_name].values
            plt.bar(x, vals, bottom=bottom, label=dec_name)
            bottom += vals
        plt.xticks(x, pivot.index.tolist(), rotation=60, ha="right")
        plt.ylabel("Nº decisiones registradas")
        plt.title("Figura 6.5 — DW: decisiones por periodo (RUN / NO_OP / EXCEPCION_FUENTE)")
        plt.legend()
        safe_savefig("fig_6_5_dw_decisiones_por_periodo.png")


{'dw.etl_control_cargas': True, 'dw.etl_decisiones_periodo': True}
OK -> outputs/EDA_GOV_INBOX_STG/tabla_dw_control_cargas.csv (24 filas)


,entidad,periodo,estado,archivo,filas_stg,filas_dw,ts_inicio,ts_fin,error_msg
0,LIC,2024-09,OK,LIC_2024-09.csv,178091,147755,2026-01-16 13:41:27.271544+00:00,2026-01-16 13:41:27.271544+00:00,None
1,OC,2024-09,OK,OC_2024-09.csv,396516,324419,2026-01-16 13:41:27.271544+00:00,2026-01-16 13:41:27.271544+00:00,None
2,LIC,2024-10,OK,LIC_2024-10.csv,219827,199155,2026-01-16 15:10:54.958475+00:00,2026-01-16 15:22:52.459721+00:00,None
3,OC,2024-10,OK,OC_2024-10.csv,508095,477995,2026-01-16 15:10:54.958475+00:00,2026-01-16 17:07:11.156930+00:00,None
4,LIC,2024-11,OK,LIC_2024-11.csv,530356,475782,2026-01-17 01:55:24.006315+00:00,2026-01-17 15:37:51.834740+00:00,None
5,OC,2024-11,OK,OC_2024-11.csv,470710,429335,2026-01-17 01:55:24.012577+00:00,2026-01-17 15:37:51.841797+00:00,None
6,LIC,2024-12,OK,LIC_2024-12.csv,359662,312983,2026-01-17 17:20:17.988311+00:00,2026-01-17 17:20:17.988311+00:00,None
7,OC,2024-12,OK,OC_2024-12.csv,411596,373131,2026-01-17 17:20:17.995190+00:00,2026-01-17 17:20:17.995190+00:00,None
8,LIC,2025-02,OK,LIC_2025-02.csv,598982,452154,2026-01-20 16:47:32.482900+00:00,2026-01-20 16:47:32.482900+00:00,None
9,OC,2025-02,OK,OC_2025-02.csv,354933,316147,2026-01-20 16:47:32.491460+00:00,2026-01-20 16:47:32.491460+00:00,None


OK -> outputs/EDA_GOV_INBOX_STG/tabla_dw_decisiones_periodo.csv (4 filas)


,periodo,decision,reason_code,reason_text,lic_file,oc_file,lic_cols,oc_bytes,oc_lines,ts
0,2025-01,EXCEPCION_FUENTE,LIC_SCHEMA_ANOM,LIC_SCHEMA_ANOM,/data/ChileCompraDL/02_INBOX/LIC/2025-01/LIC_2...,/data/ChileCompraDL/02_INBOX/OC/2025-01/OC_202...,4,613494186,2006096,2026-01-20 16:14:30.483819
1,2025-03,NO_OP,OC_SIZE_ANOM,OC_SIZE_ANOM,/data/ChileCompraDL/02_INBOX/LIC/2025-03/LIC_2...,/data/ChileCompraDL/02_INBOX/OC/2025-03/OC_202...,105,13179,187,2026-01-20 16:06:56.960405
2,2025-04,NO_OP,MISSING_DIR,MISSING_DIR,/data/ChileCompraDL/02_INBOX/LIC/2025-04/LIC_2...,/data/ChileCompraDL/02_INBOX/OC/2025-04/OC_202...,0,0,0,2026-01-20 18:14:20.812247
3,2025-11,NO_OP,MISSING_FILE,MISSING_FILE,/data/ChileCompraDL/02_INBOX/LIC/2025-11/LIC_2...,/data/ChileCompraDL/02_INBOX/OC/2025-11/OC_202...,0,0,0,2026-02-19 18:56:48.587731


OK -> outputs/EDA_GOV_INBOX_STG/fig_6_5_dw_decisiones_por_periodo.png


# 4 — Texto listo para pegar en el informe (Capítulo 6)

Este notebook aporta evidencia compacta y verificable de gobernanza de datos mediante:
(i) recepción controlada en INBOX por periodo y dominio fileciteturn2file5L33-L36,
(ii) staging mensual dinámico STG fileciteturn2file5L39-L44, y
(iii) tablas oficiales de control y decisiones del pipeline fileciteturn2file5L53-L58.

El uso explícito de decisiones RUN / NO-OP / EXCEPCION_FUENTE se interpreta como gobernanza (no como fallo) fileciteturn2file5L20-L25.


# 5 — Evidencia adicional para el informe (escala + valor analítico, sin duplicar BI)

**Objetivo:** aprovechar que el entorno local tiene Postgres histórico completo para generar **gráficos académicos** que:
- cuantifican **escala** (crecimiento, tamaños físicos),
- refuerzan decisiones metodológicas del TFM (**segmentación por moneda_norm**),
- aportan **insights agregados** sin reemplazar el dashboard Power BI.

**Reglas del TFM:**
- No mezclar montos entre monedas (prohibido sumar CLP+USD+...).
- No modificar datos: solo lectura.
- Exportar CSV/PNG a `outputs/EDA_GOV_INBOX_STG/`.


In [21]:
# =========================================
# 5 - Helpers para consultas robustas (detecta columnas)
# =========================================
from sqlalchemy import inspect

def get_cols(schema: str, name: str) -> list:
    """Devuelve columnas de una tabla/vista usando INFORMATION_SCHEMA (robusto)."""
    q = """
    SELECT column_name
    FROM information_schema.columns
    WHERE table_schema=:s AND table_name=:t
    ORDER BY ordinal_position
    """
    df = read_sql_df(q, {"s": schema, "t": name})
    return df["column_name"].tolist()

def pick_first(cols: list, candidates: list):
    """Elige el primer candidato existente en cols, si no retorna None."""
    cols_lower = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_lower:
            return cols_lower[cand.lower()]
    return None

def ensure_fig_saved(fig_name: str, title: str):
    plt.title(title)
    safe_savefig(fig_name)

def df_or_status(status: str, extra: dict=None):
    d = {"status": status}
    if extra: d.update(extra)
    return pd.DataFrame([d])

# Objetos SEM (contrato analítico)
SEM_LIC = ("dw_sem", "v_fact_licitaciones_sem_v2")
SEM_OC  = ("dw_sem", "v_fact_ordenes_compra_sem_v2")

cols_lic = get_cols(*SEM_LIC)
cols_oc  = get_cols(*SEM_OC)

safe_to_csv(pd.DataFrame({"col_lic": cols_lic}), "tabla_cols_sem_lic.csv")
safe_to_csv(pd.DataFrame({"col_oc": cols_oc}), "tabla_cols_sem_oc.csv")

print("SEM LIC cols:", len(cols_lic), "SEM OC cols:", len(cols_oc))


OK -> outputs/EDA_GOV_INBOX_STG/tabla_cols_sem_lic.csv (28 filas)
OK -> outputs/EDA_GOV_INBOX_STG/tabla_cols_sem_oc.csv (25 filas)
SEM LIC cols: 28 SEM OC cols: 25


## 5.1 — Crecimiento temporal (volumetría analítica) por mes

**Figura 7.1** — Recuento mensual (LIC vs OC) + acumulado (dos líneas).  
Esto no sustituye BI: evidencia **escala y continuidad** del universo analítico final.

Exporta:
- `tabla_sem_registros_por_mes.csv`
- `fig_7_1_sem_registros_y_acumulado.png`


In [22]:
# =========================================
# 5.1 - SEM: registros por mes + acumulado
# =========================================
period_cands = ["periodo", "anio_mes", "año_mes", "year_month", "fecha_mes", "mes", "periodo_mes"]
lic_period = pick_first(cols_lic, period_cands)
oc_period  = pick_first(cols_oc,  period_cands)

if not lic_period or not oc_period:
    df_status = df_or_status("NO_CONSTA_COL_PERIODO", {
        "lic_period": lic_period, "oc_period": oc_period,
        "hint": "Agregar/normalizar columna periodo (YYYY-MM) en dw_sem para análisis temporal reproducible."
    })
    safe_to_csv(df_status, "tabla_sem_registros_por_mes_status.csv")
    display(df_status)
else:
    q = f"""
    WITH lic AS (
      SELECT {lic_period}::text AS periodo, COUNT(*)::bigint AS n_lic
      FROM {SEM_LIC[0]}.{SEM_LIC[1]}
      GROUP BY 1
    ),
    oc AS (
      SELECT {oc_period}::text AS periodo, COUNT(*)::bigint AS n_oc
      FROM {SEM_OC[0]}.{SEM_OC[1]}
      GROUP BY 1
    )
    SELECT COALESCE(lic.periodo, oc.periodo) AS periodo,
           COALESCE(n_lic, 0) AS n_lic,
           COALESCE(n_oc, 0)  AS n_oc
    FROM lic
    FULL OUTER JOIN oc USING (periodo)
    ORDER BY 1
    """
    df_mes = read_sql_df(q)
    df_mes["n_total"] = df_mes["n_lic"] + df_mes["n_oc"]
    df_mes["acum_lic"] = df_mes["n_lic"].cumsum()
    df_mes["acum_oc"]  = df_mes["n_oc"].cumsum()
    safe_to_csv(df_mes, "tabla_sem_registros_por_mes.csv")
    display(df_mes.head(20))

    x = np.arange(len(df_mes["periodo"]))
    w = 0.42
    plt.figure(figsize=(13,5))
    plt.bar(x-w/2, df_mes["n_lic"].values, width=w, label="LIC (mes)")
    plt.bar(x+w/2, df_mes["n_oc"].values,  width=w, label="OC (mes)")
    plt.plot(x, df_mes["acum_lic"].values, label="LIC acumulado")
    plt.plot(x, df_mes["acum_oc"].values,  label="OC acumulado")
    plt.xticks(x, df_mes["periodo"].tolist(), rotation=60, ha="right")
    plt.ylabel("Registros")
    plt.legend()
    ensure_fig_saved("fig_7_1_sem_registros_y_acumulado.png",
                     "Figura 7.1 — SEM: registros por mes (LIC vs OC) y acumulado")


OK -> outputs/EDA_GOV_INBOX_STG/tabla_sem_registros_por_mes.csv (13 filas)


,periodo,n_lic,n_oc,n_total,acum_lic,acum_oc
0,2024-09,147755,324419,472174,147755,324419
1,2024-10,199155,477995,677150,346910,802414
2,2024-11,475782,429335,905117,822692,1231749
3,2024-12,312983,373131,686114,1135675,1604880
4,2025-02,452154,316147,768301,1587829,1921027
5,2025-03,485478,0,485478,2073307,1921027
6,2025-04,494536,405677,900213,2567843,2326704
7,2025-05,470000,376060,846060,3037843,2702764
8,2025-06,407938,372131,780069,3445781,3074895
9,2025-07,372640,382670,755310,3818421,3457565


OK -> outputs/EDA_GOV_INBOX_STG/fig_7_1_sem_registros_y_acumulado.png


## 5.2 — Tamaño físico por schema (stg / dw / dw_sem)

**Figura 7.2** — Tamaño (MB) por schema (tablas de usuario).  
Aporta evidencia de **infraestructura real** y separación formal de capas.

Exporta:
- `tabla_size_schema_mb.csv`
- `fig_7_2_size_schema_mb.png`


In [23]:
# =========================================
# 5.2 - Size por schema (MB)
# =========================================
q = """
SELECT
  schemaname,
  SUM(pg_total_relation_size(relid)) / 1024.0 / 1024.0 AS size_mb
FROM pg_catalog.pg_statio_user_tables
GROUP BY schemaname
ORDER BY size_mb DESC;
"""
df_size = read_sql_df(q)
safe_to_csv(df_size, "tabla_size_schema_mb.csv")
display(df_size.head(30))

top = df_size.head(10).copy()
plt.figure(figsize=(10,5))
plt.bar(top["schemaname"].astype(str), top["size_mb"].astype(float))
plt.xticks(rotation=45, ha="right")
plt.ylabel("MB")
ensure_fig_saved("fig_7_2_size_schema_mb.png", "Figura 7.2 — Tamaño físico por schema (MB) — Top 10")


OK -> outputs/EDA_GOV_INBOX_STG/tabla_size_schema_mb.csv (2 filas)


,schemaname,size_mb
0,public,18373.296875
1,dw,4349.554688


OK -> outputs/EDA_GOV_INBOX_STG/fig_7_2_size_schema_mb.png


## 5.3 — Segmentación por moneda_norm (evidencia para BI)

**Figura 8.1 (LIC)** y **Figura 8.2 (OC)** — Recuento por `moneda_norm` (no monetario).  

Exports:
- `tabla_sem_conteo_por_moneda_lic.csv`, `fig_8_1_conteo_moneda_lic.png`
- `tabla_sem_conteo_por_moneda_oc.csv`,  `fig_8_2_conteo_moneda_oc.png`


In [24]:
# =========================================
# 5.3 - Conteo por moneda_norm
# =========================================
moneda_col_lic = pick_first(cols_lic, ["moneda_norm", "moneda"])
moneda_col_oc  = pick_first(cols_oc,  ["moneda_norm", "moneda"])

def plot_counts_by_moneda(schema, view, moneda_col, out_csv, out_png, title):
    if not moneda_col:
        df_status = df_or_status("NO_CONSTA_COL_MONEDA", {"view": f"{schema}.{view}"})
        safe_to_csv(df_status, out_csv.replace(".csv","_status.csv"))
        display(df_status)
        return
    df = read_sql_df(f"""
        SELECT COALESCE({moneda_col}::text, 'NO_CONSTA') AS moneda_norm, COUNT(*)::bigint AS n
        FROM {schema}.{view}
        GROUP BY 1
        ORDER BY n DESC
    """)
    safe_to_csv(df, out_csv)
    # Limpieza para plotting (matplotlib no acepta None en etiquetas)
    df["moneda_norm"] = df["moneda_norm"].fillna("NO_CONSTA").astype(str)
    df["n"] = pd.to_numeric(df["n"], errors="coerce").fillna(0)
    plt.figure(figsize=(8,4))
    plt.bar(df["moneda_norm"], df["n"])
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("Nº registros")
    ensure_fig_saved(out_png, title)

plot_counts_by_moneda(*SEM_LIC, moneda_col_lic,
                      "tabla_sem_conteo_por_moneda_lic.csv",
                      "fig_8_1_conteo_moneda_lic.png",
                      "Figura 8.1 — LIC (SEM): conteo por moneda_norm")

plot_counts_by_moneda(*SEM_OC, moneda_col_oc,
                      "tabla_sem_conteo_por_moneda_oc.csv",
                      "fig_8_2_conteo_moneda_oc.png",
                      "Figura 8.2 — OC (SEM): conteo por moneda_norm")


OK -> outputs/EDA_GOV_INBOX_STG/tabla_sem_conteo_por_moneda_lic.csv (4 filas)
OK -> outputs/EDA_GOV_INBOX_STG/fig_8_1_conteo_moneda_lic.png
OK -> outputs/EDA_GOV_INBOX_STG/tabla_sem_conteo_por_moneda_oc.csv (5 filas)
OK -> outputs/EDA_GOV_INBOX_STG/fig_8_2_conteo_moneda_oc.png


## 5.4 — Boxplot de montos por moneda_norm (si existe monto)

**Figura 8.3 (LIC)** y **Figura 8.4 (OC)** — Boxplot por moneda_norm.  
Usa muestreo controlado (LIMIT por moneda).

Exports:
- `tabla_percentiles_montos_lic.csv`, `fig_8_3_boxplot_montos_lic.png`
- `tabla_percentiles_montos_oc.csv`,  `fig_8_4_boxplot_montos_oc.png`


In [25]:
# =========================================
# 5.4 - Montos: percentiles + boxplot por moneda (muestreo)
# =========================================
monto_cands = ["monto_total", "monto", "monto_final", "monto_adjudicado", "monto_oc", "monto_lic"]
lic_monto = pick_first(cols_lic, monto_cands)
oc_monto  = pick_first(cols_oc,  monto_cands)

def percentiles_por_moneda(schema, view, moneda_col, monto_col, out_csv):
    if not moneda_col or not monto_col:
        df_status = df_or_status("NO_CONSTA_COL_MONEDA_O_MONTO", {
            "view": f"{schema}.{view}", "moneda_col": moneda_col, "monto_col": monto_col
        })
        safe_to_csv(df_status, out_csv.replace(".csv","_status.csv"))
        display(df_status)
        return None
    q = f"""
    SELECT
      {moneda_col}::text AS moneda_norm,
      COUNT(*)::bigint AS n,
      AVG({monto_col})::numeric AS avg_monto,
      STDDEV_POP({monto_col})::numeric AS std_monto,
      PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY {monto_col}) AS p50,
      PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY {monto_col}) AS p90,
      PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY {monto_col}) AS p95,
      PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY {monto_col}) AS p99,
      MAX({monto_col}) AS max_monto
    FROM {schema}.{view}
    WHERE {monto_col} IS NOT NULL
    GROUP BY 1
    ORDER BY 1
    """
    df = read_sql_df(q)
    safe_to_csv(df, out_csv)
    display(df)
    return df

def boxplot_montos(schema, view, moneda_col, monto_col, out_png, sample_per_currency=20000):
    if not moneda_col or not monto_col:
        return
    monedas = read_sql_df(f"SELECT DISTINCT {moneda_col}::text AS moneda_norm FROM {schema}.{view} WHERE {moneda_col} IS NOT NULL").iloc[:,0].tolist()
    data = []
    labels = []
    for mon in monedas:
        dfm = read_sql_df(f"""
            SELECT {monto_col}::double precision AS monto
            FROM {schema}.{view}
            WHERE {moneda_col}::text = :m AND {monto_col} IS NOT NULL
            LIMIT {int(sample_per_currency)}
        """, {"m": mon})
        if len(dfm):
            data.append(dfm["monto"].values)
            labels.append(mon)
    if not data:
        return
    plt.figure(figsize=(10,4))
    plt.boxplot(data, labels=labels, showfliers=True)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("Monto")
    ensure_fig_saved(out_png, f"Boxplot de montos por moneda_norm — {schema}.{view}")

# LIC
percentiles_por_moneda(*SEM_LIC, moneda_col_lic, lic_monto, "tabla_percentiles_montos_lic.csv")
boxplot_montos(*SEM_LIC, moneda_col_lic, lic_monto, "fig_8_3_boxplot_montos_lic.png")

# OC
percentiles_por_moneda(*SEM_OC, moneda_col_oc, oc_monto, "tabla_percentiles_montos_oc.csv")
boxplot_montos(*SEM_OC, moneda_col_oc, oc_monto, "fig_8_4_boxplot_montos_oc.png")


OK -> outputs/EDA_GOV_INBOX_STG/tabla_percentiles_montos_lic.csv (4 filas)


,moneda_norm,n,avg_monto,std_monto,p50,p90,p95,p99,max_monto
0,CLF,3545,3.156597e+07,1.037575e+09,2000.0,20439.0,51582.0,4.470588e+07,3.568134e+10
1,CLP,4025579,1.165082e+08,4.229899e+09,14824438.0,214291450.0,326539040.0,1.606701e+09,3.357381e+12
2,USD,6222,2.626140e+06,1.038044e+07,42449.0,3725100.0,21965068.0,3.475522e+07,2.099572e+08
3,None,441,3.731126e+06,4.089167e+07,4345.0,232566.0,809276.0,4.578324e+06,4.956160e+08


OK -> outputs/EDA_GOV_INBOX_STG/fig_8_3_boxplot_montos_lic.png
OK -> outputs/EDA_GOV_INBOX_STG/tabla_percentiles_montos_oc.csv (5 filas)


,moneda_norm,n,avg_monto,std_monto,p50,p90,p95,p99,max_monto
0,CLF,11901,4.815309e+03,1.268430e+05,107.10,1695.020,5.088440e+03,2.225881e+04,7.278624e+06
1,CLP,4508018,6.133929e+06,3.116620e+08,1218465.99,8886013.693,1.886388e+07,6.735281e+07,4.035709e+11
2,EUR,35,8.303711e+04,3.302194e+05,5618.83,103391.708,2.519980e+05,1.394014e+06,1.974970e+06
3,USD,12744,7.503523e+04,7.542403e+05,6240.00,77637.686,2.009160e+05,1.281130e+06,3.732732e+07
4,UTM,818,1.292916e+04,2.300330e+05,65.31,431.280,2.794858e+03,8.511743e+04,5.708406e+06


OK -> outputs/EDA_GOV_INBOX_STG/fig_8_4_boxplot_montos_oc.png


## 5.5 — Concentración institucional (Top 10 por conteo)

Aporta insight defendible sin mezclar monedas (usa conteo).

- **Figura 8.5 (LIC)** Top 10 organismos por nº licitaciones
- **Figura 8.6 (OC)** Top 10 organismos por nº órdenes de compra

Exports:
- `tabla_top10_organismos_lic.csv`, `fig_8_5_top10_organismos_lic.png`
- `tabla_top10_organismos_oc.csv`,  `fig_8_6_top10_organismos_oc.png`


In [26]:
# =========================================
# 5.5 - Top 10 organismos (modelo dimensional correcto)
# =========================================
# Dimensión confirmada por INFORMATION_SCHEMA:
# dw_sem.v_dim_organismo_sem_v2 contiene 'nombre_organismo' (no 'organismo_nombre').

# LIC: Top 10 organismos por nº de licitaciones
q_lic = """
SELECT d.nombre_organismo,
       COUNT(*)::bigint AS n
FROM dw_sem.v_fact_licitaciones_sem_v2 f
JOIN dw_sem.v_dim_organismo_sem_v2 d
  ON f.organismo_sk = d.organismo_sk
GROUP BY d.nombre_organismo
ORDER BY n DESC
LIMIT 10;
"""
df_top_lic = read_sql_df(q_lic)
safe_to_csv(df_top_lic, "tabla_top10_organismos_lic.csv")

plt.figure(figsize=(10,4))
plt.bar(df_top_lic["nombre_organismo"].fillna("NO_CONSTA").astype(str), df_top_lic["n"])
plt.xticks(rotation=60, ha="right")
plt.ylabel("Nº licitaciones")
ensure_fig_saved("fig_8_5_top10_organismos_lic.png",
                 "Figura 8.5 — LIC (SEM): Top 10 organismos por nº licitaciones")

# OC: Top 10 organismos por nº de órdenes de compra
q_oc = """
SELECT d.nombre_organismo,
       COUNT(*)::bigint AS n
FROM dw_sem.v_fact_ordenes_compra_sem_v2 f
JOIN dw_sem.v_dim_organismo_sem_v2 d
  ON f.organismo_sk = d.organismo_sk
GROUP BY d.nombre_organismo
ORDER BY n DESC
LIMIT 10;
"""
df_top_oc = read_sql_df(q_oc)
safe_to_csv(df_top_oc, "tabla_top10_organismos_oc.csv")

plt.figure(figsize=(10,4))
plt.bar(df_top_oc["nombre_organismo"].fillna("NO_CONSTA").astype(str), df_top_oc["n"])
plt.xticks(rotation=60, ha="right")
plt.ylabel("Nº órdenes de compra")
ensure_fig_saved("fig_8_6_top10_organismos_oc.png",
                 "Figura 8.6 — OC (SEM): Top 10 organismos por nº OC")


OK -> outputs/EDA_GOV_INBOX_STG/tabla_top10_organismos_lic.csv (10 filas)
OK -> outputs/EDA_GOV_INBOX_STG/fig_8_5_top10_organismos_lic.png
OK -> outputs/EDA_GOV_INBOX_STG/tabla_top10_organismos_oc.csv (10 filas)
OK -> outputs/EDA_GOV_INBOX_STG/fig_8_6_top10_organismos_oc.png


/tmp/ipykernel_200296/1743460597.py:22: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all axes decorations.
  plt.tight_layout()


In [27]:
# =========================================
# 5.6 - Reducción STG -> SEM (autónomo, sin variables implícitas)
# =========================================
# Evita errores por nombres de dataframes (df_sem, df_mes, etc.).
# Total STG se calcula desde df_stg (conteo por tabla/mes).
# Total SEM se consulta DIRECTAMENTE desde las vistas oficiales del contrato analítico.

if "df_stg" not in globals() or not isinstance(df_stg, pd.DataFrame) or "n_registros" not in df_stg.columns:
    df_status = pd.DataFrame([{
        "status":"NO_CONSTA_df_stg",
        "hint":"Ejecuta la sección STG que construye df_stg (tabla_stg_registros_por_mes) antes de este bloque."
    }])
    safe_to_csv(df_status, "tabla_reduccion_stg_sem_status.csv")
    display(df_status)
else:
    total_stg = pd.to_numeric(df_stg["n_registros"], errors="coerce").fillna(0).sum()

    q_total_sem = """
    SELECT
        (SELECT COUNT(*) FROM dw_sem.v_fact_licitaciones_sem_v2) +
        (SELECT COUNT(*) FROM dw_sem.v_fact_ordenes_compra_sem_v2)
        AS n_total_sem;
    """
    df_total_sem = read_sql_df(q_total_sem)
    total_sem = int(df_total_sem["n_total_sem"].iloc[0])

    reduccion_pct = 100.0 * (1.0 - (total_sem / total_stg if total_stg else 0.0))

    df_reduccion = pd.DataFrame([{
        "total_registros_stg": int(total_stg),
        "total_registros_sem": int(total_sem),
        "reduccion_%": round(reduccion_pct, 2)
    }])
    safe_to_csv(df_reduccion, "tabla_reduccion_stg_sem.csv")
    display(df_reduccion)


OK -> outputs/EDA_GOV_INBOX_STG/tabla_reduccion_stg_sem.csv (1 filas)


,total_registros_stg,total_registros_sem,reduccion_%
0,11031127,9189893,16.69


In [28]:
# =========================================
# 5.X - Ratio LIC / OC por mes (correcto DW)
# =========================================

q_ratio = """
WITH lic AS (
    SELECT 
        to_char(d.fecha, 'YYYY-MM') AS periodo,
        COUNT(*)::bigint AS n_lic
    FROM dw_sem.v_fact_licitaciones_sem_v2 f
    JOIN dw.dim_fecha d 
        ON f.fecha_publicacion_sk = d.fecha_sk
    GROUP BY 1
),
oc AS (
    SELECT 
        to_char(d.fecha, 'YYYY-MM') AS periodo,
        COUNT(*)::bigint AS n_oc
    FROM dw_sem.v_fact_ordenes_compra_sem_v2 f
    JOIN dw.dim_fecha d 
        ON f.fecha_creacion_sk = d.fecha_sk
    GROUP BY 1
)
SELECT 
    COALESCE(lic.periodo, oc.periodo) AS periodo,
    COALESCE(n_lic,0) AS n_lic,
    COALESCE(n_oc,0) AS n_oc,
    (COALESCE(n_lic,0)::numeric / NULLIF(COALESCE(n_oc,0),0)) AS ratio_lic_oc
FROM lic
FULL OUTER JOIN oc USING (periodo)
ORDER BY 1;
"""

df_ratio = read_sql_df(q_ratio)

safe_to_csv(df_ratio, "tabla_ratio_lic_oc_por_mes.csv")
df_ratio.head()

OK -> outputs/EDA_GOV_INBOX_STG/tabla_ratio_lic_oc_por_mes.csv (14 filas)


,periodo,n_lic,n_oc,ratio_lic_oc
0,2024-09,147755,372489,0.396669
1,2024-10,199155,487577,0.408459
2,2024-11,475782,427333,1.113375
3,2024-12,312983,321890,0.972329
4,2025-01,0,30659,0.000000


In [29]:
# =========================================
# 5.X - Comparación tamaño físico por capa
# =========================================

plt.figure(figsize=(10,4))
plt.plot(df_ratio["periodo"].astype(str), 
         pd.to_numeric(df_ratio["ratio_lic_oc"], errors="coerce"))
plt.xticks(rotation=60, ha="right")
plt.ylabel("Ratio LIC / OC")
ensure_fig_saved("fig_ratio_lic_oc.png",
                 "Figura — Ratio LIC / OC por periodo (SEM)")

OK -> outputs/EDA_GOV_INBOX_STG/fig_ratio_lic_oc.png


In [30]:
# =========================================
# 5.X - Resumen ejecutivo cuantitativo final (robusto)
# =========================================

# --- Total registros SEM ---
q_total_sem = """
SELECT
    (SELECT COUNT(*) FROM dw_sem.v_fact_licitaciones_sem_v2) +
    (SELECT COUNT(*) FROM dw_sem.v_fact_ordenes_compra_sem_v2)
    AS total_sem;
"""
df_total_sem = read_sql_df(q_total_sem)
total_sem = int(df_total_sem["total_sem"].iloc[0])

# --- Meses cubiertos (vía dim_fecha) ---
q_meses = """
WITH fechas AS (
    SELECT d.fecha
    FROM dw_sem.v_fact_licitaciones_sem_v2 f
    JOIN dw.dim_fecha d ON f.fecha_publicacion_sk = d.fecha_sk
    UNION
    SELECT d.fecha
    FROM dw_sem.v_fact_ordenes_compra_sem_v2 f
    JOIN dw.dim_fecha d ON f.fecha_creacion_sk = d.fecha_sk
)
SELECT COUNT(DISTINCT to_char(fecha, 'YYYY-MM')) AS meses_cubiertos
FROM fechas;
"""
df_meses = read_sql_df(q_meses)
meses_cubiertos = int(df_meses["meses_cubiertos"].iloc[0])

# --- Tamaño físico por schema ---
tam_stg = None
tam_dw = None

if "df_size" in globals() and isinstance(df_size, pd.DataFrame):
    if "public" in df_size["schemaname"].values:
        tam_stg = float(df_size.loc[df_size["schemaname"]=="public","size_mb"].iloc[0])
    if "dw" in df_size["schemaname"].values:
        tam_dw = float(df_size.loc[df_size["schemaname"]=="dw","size_mb"].iloc[0])

df_resumen = pd.DataFrame([{
    "meses_cubiertos": meses_cubiertos,
    "registros_totales_sem": total_sem,
    "tamano_stg_mb": tam_stg,
    "tamano_dw_mb": tam_dw
}])

safe_to_csv(df_resumen, "tabla_resumen_ejecutivo.csv")
df_resumen

OK -> outputs/EDA_GOV_INBOX_STG/tabla_resumen_ejecutivo.csv (1 filas)


,meses_cubiertos,registros_totales_sem,tamano_stg_mb,tamano_dw_mb
0,14,9189893,18373.296875,4349.554688


# 6 — Párrafos listos para informe (escala + justificación analítica)

**Párrafo A — Evidencia de escala y continuidad**  
La ejecución sobre el entorno histórico completo permitió cuantificar la continuidad temporal y la magnitud real del universo analítico (SEM). Los gráficos de volumetría mensual y crecimiento acumulado (Figuras 7.1–7.2) proporcionan evidencia verificable de escalabilidad y refuerzan que la plataforma no opera sobre un subconjunto reducido, sino sobre un dataset histórico consolidado y gobernado.

**Párrafo B — Justificación de segmentación por moneda y robustez para BI**  
Los resultados por `moneda_norm` (Figuras 8.1–8.4) confirman la heterogeneidad del comportamiento de montos por moneda y justifican la decisión metodológica de segmentar el análisis, evitando agregaciones inválidas entre CLP, USD y otras monedas. En consecuencia, estos outputs respaldan el diseño de KPIs en Power BI, la interpretación de outliers y la validación previa del contrato analítico consumido por el dashboard.


### Texto UCM adicional para el informe (complemento)

**Párrafo C — Evidencia cuantitativa adicional (escala y depuración)**  
Adicionalmente, se incorporan métricas cuantitativas que permiten evaluar la consolidación del universo analítico: la razón de reducción entre staging mensual (STG) y capa semántica (SEM) cuantifica el efecto de depuración e integración del pipeline, mientras que el ratio LIC/OC por período permite describir el comportamiento relativo entre dominios sin sustituir el dashboard. Estos elementos aportan evidencia verificable de escalabilidad y consistencia del contrato analítico previo al consumo en Power BI.


In [31]:
# =========================================
# 5.6 - Ratio LIC/OC por mes (robusto) — Ratio LIC/OC por mes (robusto)
# =========================================
# Las facts SEM usan *_sk de fecha. Para métricas por mes, se hace JOIN a dw.dim_fecha.
# Este bloque intenta detectar una columna de fecha en dw.dim_fecha y construir 'periodo' = YYYY-MM.

df_fecha_cols = read_sql_df("""
SELECT column_name
FROM information_schema.columns
WHERE table_schema='dw' AND table_name='dim_fecha'
ORDER BY ordinal_position;
""")
fecha_cols = df_fecha_cols['column_name'].tolist()

def pick_first(cols, cands):
    s = {c.lower(): c for c in cols}
    for x in cands:
        if x.lower() in s:
            return s[x.lower()]
    return None

fecha_key = pick_first(fecha_cols, ["fecha_sk","date_sk","sk_fecha"])
fecha_date = pick_first(fecha_cols, ["fecha","date","dt","fecha_date"])
fecha_anio = pick_first(fecha_cols, ["anio","year"])
fecha_mesn = pick_first(fecha_cols, ["mes","month"])
fecha_aniomes = pick_first(fecha_cols, ["anio_mes","año_mes","year_month","periodo","periodo_mes"])

if not fecha_key:
    df_status = pd.DataFrame([{"status":"NO_CONSTA_fecha_sk_dim_fecha",
                               "hint":"dw.dim_fecha debe exponer la PK (ej. fecha_sk) para poder agrupar por mes."}])
    safe_to_csv(df_status, "tabla_ratio_lic_oc_por_mes_status.csv")
    display(df_status)
else:
    if fecha_aniomes:
        periodo_expr = f"d.{fecha_aniomes}::text"
    elif fecha_date:
        periodo_expr = f"to_char(d.{fecha_date}, 'YYYY-MM')"
    elif fecha_anio and fecha_mesn:
        periodo_expr = f"(d.{fecha_anio}::text || '-' || lpad(d.{fecha_mesn}::text, 2, '0'))"
    else:
        periodo_expr = None
        df_status = pd.DataFrame([{"status":"NO_CONSTA_cols_periodo_dim_fecha",
                                   "hint":"No se halló anio_mes/fecha/anio+mes en dw.dim_fecha."}])
        safe_to_csv(df_status, "tabla_ratio_lic_oc_por_mes_status.csv")
        display(df_status)

    if periodo_expr:
        q = f"""
        WITH lic AS (
            SELECT {periodo_expr} AS periodo, COUNT(*)::bigint AS n_lic
            FROM dw_sem.v_fact_licitaciones_sem_v2 f
            JOIN dw.dim_fecha d ON f.fecha_publicacion_sk = d.{fecha_key}
            GROUP BY 1
        ),
        oc AS (
            SELECT {periodo_expr} AS periodo, COUNT(*)::bigint AS n_oc
            FROM dw_sem.v_fact_ordenes_compra_sem_v2 f
            JOIN dw.dim_fecha d ON f.fecha_creacion_sk = d.{fecha_key}
            GROUP BY 1
        )
        SELECT COALESCE(lic.periodo, oc.periodo) AS periodo,
               COALESCE(n_lic,0) AS n_lic,
               COALESCE(n_oc,0)  AS n_oc,
               (COALESCE(n_lic,0)::numeric / NULLIF(COALESCE(n_oc,0),0)) AS ratio_lic_oc
        FROM lic
        FULL OUTER JOIN oc USING (periodo)
        ORDER BY 1;
        """
        df_ratio = read_sql_df(q)
        safe_to_csv(df_ratio, "tabla_ratio_lic_oc_por_mes.csv")
        display(df_ratio.head(20))

        plt.figure(figsize=(10,4))
        plt.plot(df_ratio["periodo"].astype(str), pd.to_numeric(df_ratio["ratio_lic_oc"], errors="coerce"))
        plt.xticks(rotation=60, ha="right")
        plt.ylabel("Ratio LIC / OC")
        ensure_fig_saved("fig_ratio_lic_oc.png", "Figura 8.7 — Ratio LIC/OC por periodo (SEM)")


OK -> outputs/EDA_GOV_INBOX_STG/tabla_ratio_lic_oc_por_mes.csv (14 filas)


,periodo,n_lic,n_oc,ratio_lic_oc
0,2024-09,147755,372489,0.396669
1,2024-10,199155,487577,0.408459
2,2024-11,475782,427333,1.113375
3,2024-12,312983,321890,0.972329
4,2025-01,0,30659,0.000000
5,2025-02,452154,288623,1.566590
6,2025-03,485478,35075,13.841140
7,2025-04,494536,404011,1.224066
8,2025-05,470000,380491,1.235246
9,2025-06,407938,370198,1.101945


OK -> outputs/EDA_GOV_INBOX_STG/fig_ratio_lic_oc.png
